# DEMO 13: Creating a Dashboard and Adding Datasets

In Power BI, you start by opening Power BI Desktop, connecting to data via Power Query, shaping your semantic model, then building visuals. The data model and reports are tightly coupled in a `.pbix` file.

In Databricks AI/BI Dashboards, the workflow is:
1. **Create a dashboard** (browser-based, no desktop tool)
2. **Add datasets** in the Data tab (SQL queries or direct table references)
3. **Build visualizations** on the Canvas tab

| Power BI Step | Databricks Equivalent |
| --- | --- |
| Open Power BI Desktop | Click **+ New** > **Dashboard** |
| Get Data (Power Query) | **Data tab** > Add data source / Write SQL |
| Build semantic model | Write SQL datasets (your "model") |
| Drag fields to visuals | Canvas tab > Add widgets |
| Publish to Service | Click **Publish** button |

In this demo, we'll:
1. Create a new dashboard
2. Add a dataset from a Unity Catalog table
3. Write custom SQL datasets for specific chart types
4. Learn the 5 essential SQL patterns for dashboard widgets

> **Note:** Run the setup cell below to create the source tables, then follow the UI instructions.

---

In [0]:
%run ./demo_13_setup

## Part 1: Create a New Dashboard

### Step 1: Create the dashboard
1. Click the **+ New** button in the left sidebar (or use the workspace "Create" button)
2. Select **Dashboard**
3. A new untitled dashboard opens in **Draft** mode

### Step 2: Name your dashboard
1. Click the title "Untitled Dashboard" at the top left
2. Type: **Sales Performance Dashboard**
3. Press Enter

You're now in the dashboard editor with two tabs:
- **Data** tab (left panel) — where you define your datasets
- **Canvas** tab — where you build visualizations

> **Power BI parallel:** This is like opening a new `.pbix` file in Desktop. The Data tab is your Power Query + Model combined, and the Canvas is your Report view.

---

## Part 2: Add a Dataset from a Unity Catalog Table

This is the simplest way to add data — point directly at a table or view.

### Step 1: Navigate to the Data tab
1. Click the **Data** tab in the left panel (it may already be selected)
2. You'll see a blank datasets area

### Step 2: Add a data source
1. Click **+ Add data** (or the "+" icon)
2. Select **Add data source**
3. In the catalog browser, navigate to:
   - Catalog: `main`
   - Schema: `demo_dashboards_<your_username>`
   - Table: `gold_sales`
4. Click the table name to select it
5. Click **Add**

### What happens:
- A new dataset appears named `gold_sales`
- It generates a default `SELECT * FROM main.demo_dashboards_<your_username>.gold_sales` query
- You can see the query and modify it if needed

### Step 3: Preview the data
1. Click the dataset name (`gold_sales`) in the left panel
2. The SQL query editor opens showing the default `SELECT *`
3. Click **Run** (or Cmd/Ctrl + Enter) to preview results
4. You'll see all 2000 rows with all columns

> **Power BI parallel:** This is equivalent to "Get Data" > selecting a table > loading it into your model.

> **Tip:** For dashboards, you'll often want to modify the `SELECT *` to only include the columns you need, or add aggregations. We'll do that next.

---

## Part 3: Add a Custom SQL Dataset

Custom SQL gives you full control over what data a visualization receives. This is how you pre-shape data for specific chart types.

### Step 1: Create a new SQL dataset
1. In the **Data** tab, click **+ Add data**
2. Select **Create from SQL**
3. A new dataset opens with an empty SQL editor

### Step 2: Write your first dashboard query
Copy and paste this SQL into the editor:

```sql
-- Monthly Revenue Trend (for a line chart)
SELECT
  DATE_TRUNC('month', order_date) AS order_month,
  SUM(net_revenue) AS total_revenue,
  COUNT(order_id) AS order_count
FROM main.demo_dashboards_<your_username>.gold_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY order_month
```

### Step 3: Run and rename
1. Click **Run** to execute and preview results
2. Click the dataset name (e.g., "Untitled dataset") and rename it to: **Monthly Revenue Trend**

> **Important:** Replace `<your_username>` with your actual schema name (shown in the setup output).

> **Power BI parallel:** This is like writing a custom SQL query in Power Query's Advanced Editor, or creating a DAX table with calculated columns.

---

## Part 4: The 5 Essential SQL Patterns for Dashboard Widgets

Every dashboard visualization ultimately needs data in a specific shape. Here are the 5 patterns that cover 95% of dashboard use cases:

| Pattern | Widget Type | Power BI Equivalent |
| --- | --- | --- |
| Simple aggregation | Counter (KPI card) | Card visual with a DAX measure |
| Group-by + aggregation | Bar/Column chart | Clustered bar with category + measure |
| Time-series aggregation | Line/Area chart | Line chart with date axis + measure |
| Filtered aggregation | Conditional KPI | CALCULATE with filter context |
| Top-N query | Ranked table, focused chart | TopN filter in Power BI |

Let's create a dataset for each pattern. Add each one as a new **Create from SQL** dataset in your dashboard.

---

In [0]:
%sql
-- ============================================================
-- Pattern 1: Simple Aggregation (KPI Card / Counter)
-- ============================================================
-- Dataset name: "KPI Summary"
-- Widget type: Counter
-- Power BI equivalent: Card visual with SUM(Revenue) measure

-- This produces a SINGLE ROW with key metrics.
-- Each column becomes a separate counter widget.

SELECT
  ROUND(SUM(net_revenue), 0) AS total_revenue,
  COUNT(order_id) AS total_orders,
  ROUND(AVG(net_revenue), 2) AS avg_order_value,
  ROUND(SUM(net_revenue) - SUM(cost), 0) AS total_profit
FROM main.demo_dashboards_<your_username>.gold_sales
WHERE order_date >= '2024-01-01';

In [0]:
%sql
-- ============================================================
-- Pattern 2: Group-By Aggregation (Bar Chart)
-- ============================================================
-- Dataset name: "Revenue by Region"
-- Widget type: Bar chart (region on X, revenue on Y)
-- Power BI equivalent: Clustered bar chart with Region + Revenue

SELECT
  region,
  ROUND(SUM(net_revenue), 2) AS total_revenue,
  COUNT(order_id) AS order_count
FROM main.demo_dashboards_<your_username>.gold_sales
GROUP BY region
ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- Pattern 3: Time-Series Aggregation (Line Chart)
-- ============================================================
-- Dataset name: "Monthly Revenue Trend"
-- Widget type: Line chart (month on X, revenue on Y)
-- Power BI equivalent: Line chart with Date hierarchy + Revenue

-- The ORDER BY ensures the line chart plots correctly over time.

SELECT
  DATE_TRUNC('month', order_date) AS order_month,
  ROUND(SUM(net_revenue), 2) AS total_revenue,
  COUNT(order_id) AS order_count
FROM main.demo_dashboards_<your_username>.gold_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY order_month;

In [0]:
%sql
-- ============================================================
-- Pattern 4: Filtered Aggregation (Conditional KPIs)
-- ============================================================
-- Dataset name: "Channel Comparison"
-- Widget type: Counter or comparison chart
-- Power BI equivalent: CALCULATE(SUM(Revenue), Channel = "Online")

-- Compare Online vs Retail performance in a single dataset.

SELECT
  ROUND(SUM(CASE WHEN channel = 'Online' THEN net_revenue END), 0) AS online_revenue,
  ROUND(SUM(CASE WHEN channel = 'Retail Store' THEN net_revenue END), 0) AS store_revenue,
  COUNT(CASE WHEN channel = 'Online' THEN 1 END) AS online_orders,
  COUNT(CASE WHEN channel = 'Retail Store' THEN 1 END) AS store_orders
FROM main.demo_dashboards_<your_username>.gold_sales;

In [0]:
%sql
-- ============================================================
-- Pattern 5: Top-N Query (Ranked Table / Focused Chart)
-- ============================================================
-- Dataset name: "Top 10 Products"
-- Widget type: Table or horizontal bar chart
-- Power BI equivalent: TopN visual-level filter

SELECT
  product_name,
  product_category,
  ROUND(SUM(net_revenue), 2) AS total_revenue,
  SUM(quantity) AS units_sold
FROM main.demo_dashboards_<your_username>.gold_sales
GROUP BY product_name, product_category
ORDER BY total_revenue DESC
LIMIT 10;

## Dashboard SQL Best Practices

| Rule | Why | Example |
| --- | --- | --- |
| One query per dataset | Each dataset is a self-contained result set | Don't UNION unrelated data |
| Pre-aggregate when possible | Widgets re-aggregate on top of results | GROUP BY at the right grain |
| Use meaningful aliases | Column names become field names in the UI | `SUM(x) AS total_revenue` |
| Include filter-friendly columns | Filters operate on dataset columns | Keep `region`, `date`, `category` |
| ORDER BY for tables | Tables display data in your specified order | `ORDER BY revenue DESC` |
| LIMIT for safety | Tables max 64K rows; charts rarely need >1000 | `LIMIT 1000` |

### Aggregation Warning

⚠️ **Double-aggregation trap:** The chart widget applies an aggregation (SUM, AVG, COUNT) ON TOP of your query results.

- If your SQL does `SUM(revenue) GROUP BY region` and the chart also sums → it sums the sums (correct for SUM, **wrong for AVG**)
- **Rule:** Either aggregate in SQL and use SUM in chart, OR return raw detail and let the chart aggregate. Never double-aggregate averages.

---

---

## Summary

| Step | What You Did |
| --- | --- |
| Create dashboard | **+ New** > **Dashboard** (browser-based, auto-saves) |
| Add UC table | Data tab > **Add data source** > browse catalog |
| Write SQL dataset | Data tab > **Create from SQL** > write + run |
| Pattern 1 | Simple aggregation → Counter widgets |
| Pattern 2 | Group-by → Bar/Column charts |
| Pattern 3 | Time-series → Line/Area charts |
| Pattern 4 | Filtered aggregation → Conditional KPIs |
| Pattern 5 | Top-N → Ranked tables |

**Key differences from Power BI:**
- No desktop tool — everything is browser-based
- No DAX — SQL is the calculation language for datasets
- No implicit model — each dataset is an explicit SQL query
- Data stays in the lakehouse — no extraction to a separate BI engine

**Next:** We'll add visualizations to this dashboard using both the configuration panel and AI-assisted chart generation.